# Featherweight — Phase 2 / Group C: QLoRA training on Colab (T4)

Run **top to bottom** on a **GPU runtime** (Runtime → Change runtime type → T4 GPU).
This is a *thin driver*: all logic lives in the `featherweight` package; this
notebook only orchestrates the Colab-only steps.

> **Caveat (read first):** the GPU-only code below (the Unsloth `train()` call and
> the generation/eval cells) is **authored but Colab-validated** — it follows
> Unsloth's standard Llama-3.1 recipe but the exact `SFTConfig` / generation
> padding / adapter-reload details are version-sensitive. Expect to adjust a line
> or two on the first run. Everything in the package was unit-tested locally.

In [ ]:
!nvidia-smi  # confirm a T4 (Turing, SM 75 -> fp16, not bf16)

## 1. Repo + install

Unsloth's official installer manages torch/transformers/trl/peft/bitsandbytes/xformers.
The clone is **idempotent and absolute** so re-running this cell can't nest the repo
inside itself (the cause of earlier path mismatches).

In [ ]:
# Idempotent clone to an ABSOLUTE path, then cd there. Safe to re-run.
![ -d /content/Featherweight/.git ] || git clone https://github.com/Nishaant-Soni/Featherweight.git /content/Featherweight
%cd /content/Featherweight

%pip install -q unsloth mlflow      # Unsloth official install; mlflow for tracking
%pip install -q -e .                # the featherweight package (editable)

> **Editable install + a running kernel:** `pip install -e .` registers the
> package via a `.pth` that Python only reads at startup, so the *current* kernel
> won't see `featherweight` yet. The cell below adds `src/` to the path for this
> session (alternatively: Restart session and re-run from the clone cell).

In [ ]:
import sys
sys.path.insert(0, "/content/Featherweight/src")  # so the running kernel sees the package

import torch, featherweight
from featherweight import config
print("torch CUDA:", torch.cuda.is_available(), "| featherweight", featherweight.__version__)
print("ROOT_DIR:", config.ROOT_DIR)  # MUST be /content/Featherweight (no nesting)

In [ ]:
# Optional: capture the resolved GPU stack for reproducibility (commit this file).
!pip freeze > requirements-colab-train.lock.txt

## 2. Workdir, secrets, and the MLflow backend

In [ ]:
from pathlib import Path

# Prefer Google Drive for persistence; fall back to local /content if the mount
# fails (common under the VS Code Colab extension). If you land on local disk,
# /content is ephemeral - download the adapter at the end (last cell).
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/featherweight")
except Exception as e:
    print(f"Drive mount unavailable ({e}); using local /content (ephemeral).")
    DRIVE = Path("/content/featherweight_out")

DRIVE.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR = DRIVE / "adapter"
print("workdir:", DRIVE)

In [ ]:
# HF token is OPTIONAL here: the datasets and the unsloth/...bnb-4bit base are all
# ungated. A read token just avoids download rate limits. (Write token only needed
# later if you push the adapter to the Hub.)
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN set - fine (everything used here is ungated).")

In [ ]:
# MLflow 3.x errors on the default file:// store (found in Group B), so use sqlite.
import mlflow
mlflow.set_tracking_uri(f"sqlite:///{DRIVE / 'mlflow.db'}")

## 3. Regenerate the formatted data on Colab

Cheaper than uploading the 195 MB `train.jsonl`: re-run the (CPU-only) prep; the
datasets are cached after the first download. Run from the repo root so the data
lands in `/content/Featherweight/data` (where training looks).

In [ ]:
!cd /content/Featherweight && python scripts/prep_data.py

## 4. Smoke run (tiny — pipeline sanity before the full run)

In [ ]:
from featherweight.train import sft

sft.train(max_steps=10, limit=64, output_dir=DRIVE / "adapter_smoke")

## 5. Full QLoRA run (checkpoints the adapter to the workdir)

In [ ]:
sft.train(output_dir=ADAPTER_DIR)

## 6. Held-out eval: base vs fine-tuned (the Phase 2 verify)

Uses the local-tested `eval.heldout.score`. The generation helper is Colab-validated
(batched left-padded greedy decode); adjust if your stack needs it.

In [ ]:
import torch
from unsloth import FastLanguageModel

from featherweight import config
from featherweight.eval import heldout
from featherweight.utils import tracking

EVAL_N = 200
rows = heldout.load_heldout()[:EVAL_N]


def generate_and_score(model, tokenizer, rows, max_new_tokens=256, batch_size=16):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    preds = []
    for i in range(0, len(rows), batch_size):
        prompts = [r["prompt"] for r in rows[i : i + batch_size]]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens, do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        new_tokens = out[:, inputs["input_ids"].shape[1]:]
        preds += tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
    return heldout.score(preds, [r["completion"] for r in rows])


base_model, base_tok = FastLanguageModel.from_pretrained(
    config.BASE_MODEL_4BIT, max_seq_length=2048, dtype=None, load_in_4bit=True,
)
base_metrics = generate_and_score(base_model, base_tok, rows)
print("BASE:", base_metrics)

In [ ]:
# Reload the fine-tuned adapter (Unsloth loads the saved LoRA dir) and score it.
ft_model, ft_tok = FastLanguageModel.from_pretrained(
    str(ADAPTER_DIR), max_seq_length=2048, dtype=None, load_in_4bit=True,
)
ft_metrics = generate_and_score(ft_model, ft_tok, rows)
print("FT:  ", ft_metrics)

with tracking.mlflow_run("heldout-base-vs-ft"):
    tracking.log_metrics({f"base_{k}": v for k, v in base_metrics.items()})
    tracking.log_metrics({f"ft_{k}": v for k, v in ft_metrics.items()})

## 7. Save the adapter

If you trained to local `/content` (Drive mount failed), download the adapter so a
disconnect doesn't lose it. If you used Drive, it is already persisted there.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/featherweight_adapter", "zip", root_dir=ADAPTER_DIR)
print("zipped:", zip_path)
files.download(zip_path)

## Verify (Phase 2 done when all hold)

- [ ] Full run completed within the T4 VRAM budget (`nvidia-smi` peak < ~10 GB).
- [ ] Training loss decreased (MLflow).
- [ ] `ft_metrics["exact_match_accuracy"]` beats `base_metrics["exact_match_accuracy"]`
      by a clear margin; refusal accuracy improved.
- [ ] Adapter saved (Drive) or downloaded (local-disk fallback).